<a href="https://colab.research.google.com/github/marinhomar/cardioia-fase2/blob/main/parte1_extracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Bibliotecas

In [9]:
# pandas: para trabalhar com tabelas (como uma planilha dentro do Python)
import pandas as pd

# unicodedata: para tirar os acentos das palavras
import unicodedata

## 2. Leitura dos arquivos (frases, mapa e sinônimos)

In [ ]:
# endereço da pasta onde estão os arquivos no GitHub
BASE = "https://raw.githubusercontent.com/marinhomar/cardioia-fase2/main/src/parte1/"

# frases dos pacientes (uma por linha)
# sep="\t" porque as frases têm vírgula e não podem ser cortadas
frases = pd.read_csv(BASE + "frases.txt", header=None, names=["frase"], sep="\t")

# mapa: qual sintoma aponta para qual doença
mapa = pd.read_csv(BASE + "mapa_conhecimento.csv")

# sinônimos: jeitos populares de falar o mesmo sintoma
sinonimos = pd.read_csv(BASE + "sinonimos.csv")

# mostra as frases para conferir se carregou certo
frases

## 3. Padronização do texto

In [4]:
# deixa o texto em minúsculo e sem acento
# assim "Tórax", "torax" e "TÓRAX" viram a mesma coisa
def normalizar(texto):
    texto = texto.lower()                          # tudo minúsculo
    texto = unicodedata.normalize("NFD", texto)    # separa a letra do acento
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")  # joga o acento fora

## 4. Sinônimos populares ( MELHORIAS )

In [5]:
# coloca os sinônimos no mesmo padrão (minúsculo e sem acento)
sinonimos["variante"] = sinonimos["variante"].apply(normalizar)
sinonimos["termo_padrao"] = sinonimos["termo_padrao"].apply(normalizar)

# ordena do maior para o menor
# assim "pernas inchada" é trocada antes de "perna inchada"
sinonimos = sinonimos.sort_values("variante", key=lambda s: s.str.len(), ascending=False)

# troca o jeito popular pelo termo que está no mapa
# exemplo: "agonia no peito" vira "dor no peito"
def traduzir(texto):
    for variante, padrao in zip(sinonimos["variante"], sinonimos["termo_padrao"]):
        texto = texto.replace(variante, padrao)
    return texto

## 5. Regra de diagnóstico e desempate por gravidade

In [6]:
# ordem de gravidade: em caso de empate, vale a mais grave (como na triagem do pronto-socorro)
GRAVIDADE = ["Infarto", "Angina", "Insuficiência Cardíaca"]

def diagnosticar(frase):
    # prepara a frase: padroniza e traduz os sinônimos
    texto = traduzir(normalizar(frase))

    pontos = {}        # quantos sintomas cada doença recebeu
    encontrados = []   # quais sintomas apareceram na frase

    # passa por cada linha do mapa procurando os sintomas na frase
    for _, linha in mapa.iterrows():
        for sintoma in (linha["Sintoma 1"], linha["Sintoma 2"]):
            if normalizar(sintoma) in texto:
                encontrados.append(sintoma)
                doenca = linha["Doença Associada"]
                pontos[doenca] = pontos.get(doenca, 0) + 1   # +1 ponto para essa doença

    # nenhum sintoma encontrado
    if not pontos:
        return encontrados, "Sem diagnóstico"

    # pega a doença (ou doenças) com mais pontos
    maior = max(pontos.values())
    empatadas = [d for d in pontos if pontos[d] == maior]

    # só uma ganhou: esse é o diagnóstico
    if len(empatadas) == 1:
        return encontrados, empatadas[0]

    # empate: avisa e sugere a mais grave
    prioridade = min(empatadas, key=GRAVIDADE.index)
    return encontrados, f"Inconclusivo ({' / '.join(empatadas)}) → priorizar: {prioridade}"

## 6. Resultado das 10 frases

In [10]:
# copia a tabela de frases para não mexer na original
resultado = frases.copy()

# aplica o diagnóstico em cada frase e cria duas colunas novas
resultado[["sintomas_encontrados", "diagnostico"]] = resultado["frase"].apply(
    lambda f: pd.Series(diagnosticar(f))
)

# deixa mostrar a frase inteira na tabela
pd.set_option("display.max_colwidth", 120)

# mostra o resultado final
resultado

,frase,sintomas_encontrados,diagnostico
0,Desde hoje cedo sinto uma dor no peito forte com suor frio e não consegui sair da cama para trabalhar.,"[dor no peito, suor frio]",Infarto
1,"Há duas horas estou com um aperto no tórax e dor no braço esquerdo, tive que parar de dirigir no meio do caminho.","[aperto no tórax, dor no braço esquerdo]",Infarto
2,"Desde a madrugada sinto dor no queixo e náusea, e não me sinto bem.","[dor no queixo, náusea]",Infarto
3,"Há três semanas sinto dor ao subir escada, mas passa quando descanso; agora evito ir ao segundo andar do escritório.","[dor ao subir escada, passa quando descanso]",Angina
4,"Há um mês tenho dor ao fazer esforço na academia e ela melhora com repouso, por isso parei de treinar.","[dor ao fazer esforço, melhora com repouso]",Angina
5,"Faz dez dias que sinto falta de ar e dificuldade para respirar quando caminho rápido, e deixei de ir a pé para o tra...","[falta de ar, dificuldade para respirar]",Angina
6,Há uma semana sinto cansaço constante mesmo depois de descansar e não consigo terminar meu dia bem.,[cansaço constante],Insuficiência Cardíaca
7,"Há quinze dias estou com as pernas inchadas e os tornozelos inchados no fim do dia, e meus sapatos não servem mais.","[pernas inchadas, tornozelos inchados]",Insuficiência Cardíaca
8,"Desde a semana passada acordo sem ar durante a noite e preciso de três travesseiros, então durmo mal e chego exausto...",[acordo sem ar],Insuficiência Cardíaca
9,"Há um mês vejo que ganho peso rápido e barriga inchada, além de muita fadiga, e já não consigo brincar com meu neto.","[fadiga, ganho peso rápido, barriga inchada]",Insuficiência Cardíaca


## 7. Teste com linguagem popular e caso de empate

In [11]:
# frase nova, com expressões do dia a dia que não estão no mapa
# os sinônimos fazem a ponte e o sistema entende mesmo assim
diagnosticar("Desde ontem tô com uma agonia no peito e suando frio, não fui trabalhar.")

(['dor no peito', 'suor frio'], 'Infarto')

In [12]:
# frase que aponta para duas doenças ao mesmo tempo
# o sistema avisa que é inconclusivo e sugere a mais grave
diagnosticar("Faz uma semana que fico sem fôlego e com uma canseira danada, não dou conta de varrer a casa.")

(['cansaço constante', 'falta de ar'],
 'Inconclusivo (Insuficiência Cardíaca / Angina) → priorizar: Angina')